# Generic Pitch Builder v4 — Client-Safe by Default

**Rule of thumb: only edit Cell 1. Everything else is machinery — do not touch.**

## What this notebook does

1. Detects your client's DESTINATION variants automatically (e.g. `FOOD LOVERS MARKET` + `FOOD LOVERS EATERY`)
2. Refuses to run if the data doesn't validate (halts on cross-check failures — no wrong numbers get shipped)
3. Produces a **client-facing HTML pitch** with hero KPIs, brand split, category positioning, customer quality, demographics, geo, monthly trend, activation opportunities, and cross-shop

## Guardrails baked in (from lessons learned 2026-07-06)

- **NEVER sums from `stg_transactions`** — always uses `int_customer_category_spend` to avoid fanout inflation
- **Always cross-validates combined against set-logic union** — halts if delta > 1%
- **Never filters combined by CATEGORY_TWO** — DESTINATIONs may live in different categories (e.g. Food Lovers Eatery is Restaurants, not Groceries — filtering by category silently drops it)
- **Uses exact Rand values** for per-customer metrics — no `R4k` / `R1k` rounding hiding real differences
- **No competitor names in client-facing decks** — only your own rank + share
- **Segment / churn / CLV are opt-in** — they're FNB-wide, not client-specific
- **Auto-detects DESTINATION variants** — prevents "I queried FOOD LOVERS MARKET and missed the Eatery"
- **Prints a scope-safety checklist** at the end of the run

## Cell 1 — Config (the ONLY cell you edit)

Read the comments. They matter.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# CLIENT CONFIG — everything you need to change is here.
# ─────────────────────────────────────────────────────────────────────────

# What you call the client (used in the deck)
CLIENT_NAME       = 'Food Lovers'

# One-line context — helps you frame the narrative
CLIENT_CONTEXT    = 'aspirational middle class + fresh food + value at price'

# Patterns for finding the client's DESTINATIONs. Cell 3 will show you
# every match. Decide which ones to include below.
DESTINATION_SEARCH_PATTERNS = ['FOOD LOVERS', 'FOODLOVERS']

# The DESTINATIONs to include. Leave empty on your first run to see
# candidates from Cell 3, then fill in with exact strings.
# Example: ['FOOD LOVERS MARKET', 'FOOD LOVERS EATERY']
DESTINATIONS      = ['FOOD LOVERS MARKET', 'FOOD LOVERS EATERY']

# CATEGORY_TWO for positioning context (rank/share vs category peers).
# Only used for the "Category positioning" section.
CATEGORY_TWO      = 'Groceries'

# GCP project
PROJECT           = 'fmn-sandbox'  # or 'fmn-production-462014'

# Show the audience as:
#   'combined' — one hero KPI block (default)
#   'split'    — each DESTINATION side-by-side
#   'both'     — combined first, then each split out
COMBINED_OR_SPLIT = 'both'

# Include the ML segment donut + activation opportunities?
# Segments are FNB-wide (not client-specific). Include only if you can
# explain that caveat, OR if this pitch is internal.
INCLUDE_SEGMENTS  = True

# Output file
OUTPUT_HTML       = f'{CLIENT_NAME.lower().replace(" ", "_")}_pitch.html'


# ─────────────────────────────────────────────────────────────────────────
# OPTIONAL SECTIONS — toggle these on/off depending on the client
# ─────────────────────────────────────────────────────────────────────────

# Loyalist cohort — only meaningful for REPEAT/FRAGMENTED categories
# (groceries, fashion, restaurants, pharmacies).
# Turn OFF for one-provider categories (telecom, vehicle service, insurance).
INCLUDE_LOYALIST_SECTION = True

# Switch opportunity — almost always safe. Shows the total category pool
# vs how much of it is loyal to the client = the winnable ground.
INCLUDE_SWITCH_SECTION = True

# Aspirational acquisition target — customers who match your ideal buyer
# profile but don't shop at the client yet. Adjust the thresholds below
# to match the client's target demographic.
INCLUDE_ACQUISITION_SECTION = True
ACQUISITION_AGE_MIN            = 26
ACQUISITION_AGE_MAX            = 45
ACQUISITION_INCOME_BANDS       = ['R23.5k-R32.5k', 'R32.5k-R56k', 'R56k+']
ACQUISITION_MIN_CATEGORY_SPEND = 12000  # rand/year — filters out casual shoppers

print('Config loaded:')
print(f'  Client:       {CLIENT_NAME}')
print(f'  Destinations: {DESTINATIONS if DESTINATIONS else "(empty — run Cell 3)"}')
print(f'  Category:     {CATEGORY_TWO}')
print(f'  Project:      {PROJECT}')
print(f'  Layout:       {COMBINED_OR_SPLIT}')
print(f'  Segments:     {"YES (adds caveat)" if INCLUDE_SEGMENTS else "NO"}')
print(f'  Output:       {OUTPUT_HTML}')
print(f'  Loyalist:     {"YES" if INCLUDE_LOYALIST_SECTION else "NO"}')
print(f'  Switch:       {"YES" if INCLUDE_SWITCH_SECTION else "NO"}')
print(f'  Acquisition:  {"YES" if INCLUDE_ACQUISITION_SECTION else "NO"}')


## Cell 2 — Setup (don't touch)

In [ ]:
# Auto-install missing deps (some BQ Studio kernels lack these)
import subprocess, sys
for pkg, imp in [
    ('google-cloud-bigquery', 'google.cloud.bigquery'),
    ('db-dtypes',             'db_dtypes'),
    ('pandas',                'pandas'),
]:
    try:
        __import__(imp)
    except ImportError:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

import html as _h
import json
import pandas as pd
from datetime import datetime
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT)

def query(sql):
    return bq.query(sql).to_dataframe()

def R(v):
    """Big-picture Rand: R4.6B, R177.9M, R4k"""
    if v is None or pd.isna(v):
        return 'N/A'
    v = float(v)
    if abs(v) >= 1e12: return f'R{v/1e12:.2f}T'
    if abs(v) >= 1e9:  return f'R{v/1e9:.2f}B'
    if abs(v) >= 1e6:  return f'R{v/1e6:.1f}M'
    if abs(v) >= 1e3:  return f'R{v/1e3:.0f}k'
    return f'R{v:,.0f}'

def R_exact(v):
    """Full-precision Rand — for per-customer metrics where rounding hides the story."""
    if v is None or pd.isna(v):
        return 'N/A'
    return f'R{float(v):,.0f}'

def N(v):
    if v is None or pd.isna(v):
        return 'N/A'
    return f'{int(v):,}'

def esc(s):
    return _h.escape(str(s))

def dest_sql_in_clause(dest_list, alias=''):
    if not dest_list:
        raise RuntimeError('DESTINATIONS is empty. Run Cell 3, then fill Cell 1.')
    upper_list = ','.join(f"'{d.upper()}'" for d in dest_list)
    col = f'{alias}.DESTINATION' if alias else 'DESTINATION'
    return f'UPPER({col}) IN ({upper_list})'

print('Setup complete. Project:', bq.project)


## Cell 3 — DESTINATION detection (run this FIRST)

Shows every DESTINATION matching your search patterns. Decide which ones to include and fill `DESTINATIONS` in Cell 1.

In [ ]:
if not DESTINATION_SEARCH_PATTERNS:
    raise RuntimeError('DESTINATION_SEARCH_PATTERNS is empty. Fill it in Cell 1.')

like_clauses = ' OR '.join(
    f"UPPER(DESTINATION) LIKE '%{p.upper()}%'" for p in DESTINATION_SEARCH_PATTERNS
)

detected = query(f"""
    SELECT DESTINATION,
           CATEGORY_TWO,
           COUNT(DISTINCT UNIQUE_ID) AS customers,
           SUM(dest_txn_count)       AS transactions,
           ROUND(SUM(dest_spend), 0) AS total_spend
    FROM `{PROJECT}.analytics.int_customer_category_spend`
    WHERE {like_clauses}
    GROUP BY DESTINATION, CATEGORY_TWO
    ORDER BY customers DESC
""")

print(f'Found {len(detected)} DESTINATIONs:')
print()
print(detected.to_string(index=False))
print()

if len(detected) == 0:
    print('⚠  No matches — check your search patterns in Cell 1.')
elif not DESTINATIONS:
    print('👉 Now fill DESTINATIONS in Cell 1 with the exact strings you want.')
else:
    print(f'✓ Cell 1 has: {DESTINATIONS}')
    missing = [d for d in DESTINATIONS if d.upper() not in detected['DESTINATION'].str.upper().values]
    if missing:
        print(f'⚠ These are in Cell 1 but not in the data: {missing}')
    unlisted = [d for d in detected['DESTINATION'] if d.upper() not in [x.upper() for x in DESTINATIONS]]
    if unlisted:
        print(f'ℹ Matches your search but NOT in your list — decide if to include:')
        for u in unlisted:
            row = detected[detected['DESTINATION'] == u].iloc[0]
            print(f'  • {u}  ({N(row["customers"])} customers, {row["CATEGORY_TWO"]})')


## Cell 4 — Data queries (validated)

Halts if `DESTINATIONS` is empty. Cross-validates combined against set-logic union — halts if delta > 1%. Pulls everything Cell 5 needs.

In [ ]:
if not DESTINATIONS:
    raise RuntimeError('DESTINATIONS is empty. Run Cell 3 first, then fill Cell 1.')

DEST_IN = dest_sql_in_clause(DESTINATIONS)
DEST_IN_T = dest_sql_in_clause(DESTINATIONS, 't')
print(f'Filter: {DEST_IN}')
print()

# 1. Combined KPIs — NO CATEGORY_TWO filter (would drop DESTINATIONs in other categories)
combined = query(f"""
    SELECT COUNT(DISTINCT UNIQUE_ID)                                        AS customers,
           SUM(dest_txn_count)                                              AS transactions,
           ROUND(SUM(dest_spend), 0)                                        AS total_spend,
           ROUND(SUM(dest_spend) / NULLIF(SUM(dest_txn_count), 0), 2)       AS avg_txn_value,
           ROUND(SUM(dest_spend) / NULLIF(COUNT(DISTINCT UNIQUE_ID), 0), 0) AS spend_per_customer
    FROM `{PROJECT}.analytics.int_customer_category_spend`
    WHERE {DEST_IN}
""").iloc[0]

# 2. Cross-validate combined against set-logic union
validate_parts = []
for d in DESTINATIONS:
    validate_parts.append(
        f"(SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` "
        f"WHERE UPPER(DESTINATION) = '{d.upper()}')"
    )
union_sql = f"""
    WITH the_union AS ({' UNION DISTINCT '.join(validate_parts)})
    SELECT COUNT(*) AS union_size FROM the_union
"""
union_size = query(union_sql).iloc[0]['union_size']
primary_size = int(combined['customers'])
delta_pct = abs(primary_size - union_size) / max(union_size, 1) * 100

print(f'Combined-vs-union validation:')
print(f'  Primary query says:   {primary_size:>10,}')
print(f'  Set-logic union says: {union_size:>10,}')
print(f'  Delta:                {delta_pct:.4f}%')

if delta_pct > 1.0:
    raise RuntimeError(
        f'❌ Combined customer base disagrees by {delta_pct:.2f}%. '
        'Refusing to build pitch on unreliable numbers.'
    )
print(f'  ✓ Validated (within 1% tolerance)')
print()

# 3. Per-DESTINATION breakdown from mart_destination_benchmarks
dest_details = query(f"""
    SELECT DESTINATION, customers, transactions, total_spend,
           avg_txn_value, spend_per_customer,
           market_share_pct, penetration_pct, avg_share_of_wallet, spend_rank
    FROM `{PROJECT}.marts.mart_destination_benchmarks`
    WHERE CATEGORY_TWO = '{CATEGORY_TWO}'
      AND UPPER(DESTINATION) IN ({",".join(f"'{d.upper()}'" for d in DESTINATIONS)})
    ORDER BY total_spend DESC
""")

# 4. Any DESTINATIONs the mart doesn't have (they live in a different category)?
mart_dests = set(dest_details['DESTINATION'].str.upper().values) if not dest_details.empty else set()
missing_dests = [d for d in DESTINATIONS if d.upper() not in mart_dests]
if missing_dests:
    print(f'ℹ These DESTINATIONs are outside CATEGORY_TWO="{CATEGORY_TWO}" — computing live:')
    for d in missing_dests:
        print(f'   • {d}')
    live_details = query(f"""
        SELECT DESTINATION,
               COUNT(DISTINCT UNIQUE_ID) AS customers,
               SUM(dest_txn_count) AS transactions,
               ROUND(SUM(dest_spend), 0) AS total_spend,
               ROUND(SUM(dest_spend) / NULLIF(SUM(dest_txn_count), 0), 2) AS avg_txn_value,
               ROUND(SUM(dest_spend) / NULLIF(COUNT(DISTINCT UNIQUE_ID), 0), 0) AS spend_per_customer
        FROM `{PROJECT}.analytics.int_customer_category_spend`
        WHERE UPPER(DESTINATION) IN ({",".join(f"'{d.upper()}'" for d in missing_dests)})
        GROUP BY DESTINATION
    """)
    for _, row in live_details.iterrows():
        new_row = {c: None for c in dest_details.columns}
        for k, v in row.items():
            new_row[k] = v
        dest_details = pd.concat([dest_details, pd.DataFrame([new_row])], ignore_index=True)

# 5. Demographics
demo = query(f"""
    WITH custs AS (
        SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
    )
    SELECT COUNT(*) AS customers,
           ROUND(AVG(c.age), 1) AS avg_age,
           COUNT(DISTINCT CASE WHEN c.gender_label='Male' THEN c.UNIQUE_ID END) AS male,
           COUNT(DISTINCT CASE WHEN c.gender_label='Female' THEN c.UNIQUE_ID END) AS female
    FROM custs f JOIN `{PROJECT}.staging.stg_customers` c USING (UNIQUE_ID)
""").iloc[0]

# 6. Income × gender
income_gender = query(f"""
    WITH custs AS (
        SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
    )
    SELECT c.income_group AS band,
           COUNT(DISTINCT CASE WHEN c.gender_label='Male' THEN c.UNIQUE_ID END) AS male,
           COUNT(DISTINCT CASE WHEN c.gender_label='Female' THEN c.UNIQUE_ID END) AS female
    FROM custs f JOIN `{PROJECT}.staging.stg_customers` c USING (UNIQUE_ID)
    WHERE c.income_group IS NOT NULL AND c.income_group <> 'Unknown'
    GROUP BY c.income_group
    ORDER BY CASE c.income_group
        WHEN 'R0-R5.5k' THEN 1 WHEN 'R5.5k-R13.5k' THEN 2 WHEN 'R13.5k-R23.5k' THEN 3
        WHEN 'R23.5k-R32.5k' THEN 4 WHEN 'R32.5k-R56k' THEN 5 WHEN 'R56k+' THEN 6 ELSE 99 END
""").to_dict('records')

# 7. Age × gender
age_gender = query(f"""
    WITH custs AS (
        SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
    )
    SELECT c.age_group AS band,
           COUNT(DISTINCT CASE WHEN c.gender_label='Male' THEN c.UNIQUE_ID END) AS male,
           COUNT(DISTINCT CASE WHEN c.gender_label='Female' THEN c.UNIQUE_ID END) AS female
    FROM custs f JOIN `{PROJECT}.staging.stg_customers` c USING (UNIQUE_ID)
    WHERE c.age_group IS NOT NULL AND c.age_group <> 'Unknown'
    GROUP BY c.age_group
    ORDER BY CASE c.age_group
        WHEN '18-25' THEN 1 WHEN '26-35' THEN 2 WHEN '36-45' THEN 3
        WHEN '46-60' THEN 4 WHEN '60+' THEN 5 ELSE 99 END
""").to_dict('records')

# 8. Geographic footprint
geo = query(f"""
    WITH custs AS (
        SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
    )
    SELECT t.PROVINCE AS province,
           COUNT(DISTINCT t.UNIQUE_ID) AS customers,
           ROUND(SUM(t.trns_amt), 0) AS spend
    FROM custs f JOIN `{PROJECT}.staging.stg_transactions` t USING (UNIQUE_ID)
    WHERE {DEST_IN_T} AND t.PROVINCE IS NOT NULL
    GROUP BY t.PROVINCE
    ORDER BY spend DESC
    LIMIT 9
""").to_dict('records')

# 9. Monthly trend
trend = query(f"""
    SELECT FORMAT_DATE('%Y-%m', t.EFF_DATE) AS month,
           COUNT(DISTINCT t.UNIQUE_ID) AS customers,
           ROUND(SUM(t.trns_amt), 0) AS spend
    FROM `{PROJECT}.staging.stg_transactions` t
    WHERE {DEST_IN_T}
    GROUP BY month ORDER BY month
""").to_dict('records')

# 10. Segments (opt-in)
segments, segment_spend = [], []
if INCLUDE_SEGMENTS:
    segments = query(f"""
        WITH custs AS (
            SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
        )
        SELECT co.segment_name AS segment,
               COUNT(*) AS customers,
               ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
        FROM custs f JOIN `{PROJECT}.marts.mart_cluster_output` co USING (UNIQUE_ID)
        GROUP BY co.segment_name ORDER BY customers DESC
    """).to_dict('records')
    segment_spend = query(f"""
        WITH activity AS (
            SELECT cs.UNIQUE_ID, SUM(cs.dest_spend) AS client_spend
            FROM `{PROJECT}.analytics.int_customer_category_spend` cs
            WHERE UPPER(cs.DESTINATION) IN ({",".join(f"'{d.upper()}'" for d in DESTINATIONS)})
            GROUP BY cs.UNIQUE_ID
        )
        SELECT co.segment_name AS segment,
               COUNT(*) AS customers,
               ROUND(SUM(a.client_spend), 0) AS client_annual_spend
        FROM activity a JOIN `{PROJECT}.marts.mart_cluster_output` co USING (UNIQUE_ID)
        GROUP BY co.segment_name
    """).to_dict('records')

# 11. Cross-shop
cross_shop = query(f"""
    WITH custs AS (
        SELECT DISTINCT UNIQUE_ID FROM `{PROJECT}.analytics.int_customer_category_spend` WHERE {DEST_IN}
    )
    SELECT cs.CATEGORY_TWO AS category,
           COUNT(DISTINCT cs.UNIQUE_ID) AS shoppers,
           ROUND(SUM(cs.dest_spend), 0) AS spend
    FROM custs f JOIN `{PROJECT}.analytics.int_customer_category_spend` cs USING (UNIQUE_ID)
    WHERE cs.CATEGORY_TWO <> '{CATEGORY_TWO}'
    GROUP BY cs.CATEGORY_TWO ORDER BY shoppers DESC LIMIT 8
""").to_dict('records')



# ─────────────────────────────────────────────────────────────────────────
# OPTIONAL SECTIONS — data queries
# ─────────────────────────────────────────────────────────────────────────

# 12. Loyalist cohort (share-of-wallet at client vs category)
loyalist = None
if INCLUDE_LOYALIST_SECTION:
    dest_list = ",".join(f"'{d.upper()}'" for d in DESTINATIONS)
    loyalist_row = query(f"""
        WITH client_and_category AS (
            SELECT
                UNIQUE_ID,
                SUM(CASE WHEN UPPER(DESTINATION) IN ({dest_list})
                         THEN dest_spend ELSE 0 END) AS client_spend,
                SUM(CASE WHEN CATEGORY_TWO = '{CATEGORY_TWO}'
                         THEN dest_spend ELSE 0 END) AS category_spend
            FROM `{PROJECT}.analytics.int_customer_category_spend`
            WHERE CATEGORY_TWO = '{CATEGORY_TWO}'
               OR UPPER(DESTINATION) IN ({dest_list})
            GROUP BY UNIQUE_ID
            HAVING SUM(CASE WHEN UPPER(DESTINATION) IN ({dest_list})
                             THEN dest_spend ELSE 0 END) > 0
        )
        SELECT
            COUNTIF(client_spend / NULLIF(category_spend + client_spend, 0) >= 0.60) AS loyalist_customers,
            ROUND(SUM(CASE WHEN client_spend / NULLIF(category_spend + client_spend, 0) >= 0.60
                            THEN client_spend ELSE 0 END), 0)                        AS loyalist_spend,
            COUNTIF(client_spend / NULLIF(category_spend + client_spend, 0) BETWEEN 0.20 AND 0.60) AS regular_customers,
            COUNTIF(client_spend / NULLIF(category_spend + client_spend, 0) < 0.20)  AS occasional_customers
        FROM client_and_category
    """).iloc[0]
    # Guardrail: skip section if fewer than 100 loyalists (calc is unreliable)
    if int(loyalist_row['loyalist_customers']) >= 100:
        loyalist = loyalist_row
        print(f'  Loyalists:         {int(loyalist["loyalist_customers"]):,}')
    else:
        print(f'  ⚠ Loyalist section skipped: only {int(loyalist_row["loyalist_customers"])} customers meet the threshold. Not a repeat-category client?')


# 13. Switch opportunity (total category vs client vs non-client pool)
switch = None
if INCLUDE_SWITCH_SECTION:
    if not CATEGORY_TWO:
        raise RuntimeError('INCLUDE_SWITCH_SECTION requires CATEGORY_TWO in Cell 1.')
    dest_list = ",".join(f"'{d.upper()}'" for d in DESTINATIONS)
    switch = query(f"""
        WITH category_customers AS (
            SELECT
                cs.UNIQUE_ID,
                SUM(cs.dest_spend) AS spend
            FROM `{PROJECT}.analytics.int_customer_category_spend` cs
            WHERE cs.CATEGORY_TWO = '{CATEGORY_TWO}'
            GROUP BY cs.UNIQUE_ID
        ),
        client_customers AS (
            SELECT DISTINCT UNIQUE_ID
            FROM `{PROJECT}.analytics.int_customer_category_spend`
            WHERE UPPER(DESTINATION) IN ({dest_list})
        )
        SELECT
            (SELECT COUNT(DISTINCT UNIQUE_ID) FROM category_customers)                                       AS category_customers,
            (SELECT ROUND(SUM(spend), 0) FROM category_customers)                                            AS category_spend,
            (SELECT COUNT(*) FROM client_customers)                                                          AS client_customers,
            (SELECT COUNT(DISTINCT UNIQUE_ID) FROM category_customers WHERE UNIQUE_ID NOT IN (SELECT UNIQUE_ID FROM client_customers)) AS non_client_customers,
            (SELECT ROUND(SUM(spend), 0) FROM category_customers WHERE UNIQUE_ID NOT IN (SELECT UNIQUE_ID FROM client_customers)) AS non_client_spend
    """).iloc[0]
    print(f'  Switch pool:       {int(switch["non_client_customers"]):,} non-client customers')


# 14. Aspirational acquisition target (lookalike-but-not-yet)
acq_target = None
if INCLUDE_ACQUISITION_SECTION:
    if not CATEGORY_TWO:
        raise RuntimeError('INCLUDE_ACQUISITION_SECTION requires CATEGORY_TWO in Cell 1.')
    dest_list = ",".join(f"'{d.upper()}'" for d in DESTINATIONS)
    income_list = ",".join(f"'{b}'" for b in ACQUISITION_INCOME_BANDS)
    acq_row = query(f"""
        WITH category_spenders AS (
            SELECT
                cs.UNIQUE_ID,
                SUM(cs.dest_spend) AS category_spend
            FROM `{PROJECT}.analytics.int_customer_category_spend` cs
            WHERE cs.CATEGORY_TWO = '{CATEGORY_TWO}'
            GROUP BY cs.UNIQUE_ID
        ),
        client_customers AS (
            SELECT DISTINCT UNIQUE_ID
            FROM `{PROJECT}.analytics.int_customer_category_spend`
            WHERE UPPER(DESTINATION) IN ({dest_list})
        )
        SELECT
            COUNT(DISTINCT g.UNIQUE_ID)     AS acq_customers,
            ROUND(SUM(g.category_spend), 0) AS acq_spend
        FROM category_spenders g
        JOIN `{PROJECT}.staging.stg_customers` c ON g.UNIQUE_ID = c.UNIQUE_ID
        WHERE g.UNIQUE_ID NOT IN (SELECT UNIQUE_ID FROM client_customers)
          AND c.age BETWEEN {ACQUISITION_AGE_MIN} AND {ACQUISITION_AGE_MAX}
          AND c.income_group IN ({income_list})
          AND g.category_spend >= {ACQUISITION_MIN_CATEGORY_SPEND}
    """).iloc[0]
    if int(acq_row['acq_customers']) >= 500:
        acq_target = acq_row
        print(f'  Acquisition pool:  {int(acq_target["acq_customers"]):,}')
    else:
        print(f'  ⚠ Acquisition section skipped: only {int(acq_row["acq_customers"])} prospects match. Loosen the age/income thresholds in Cell 1.')

print()
print('✓ All data pulled and validated.')
print(f'  Combined customers: {int(combined["customers"]):,}')
print(f'  Combined spend:     {R(combined["total_spend"])}')
print(f'  DESTINATIONs shown: {len(dest_details)}')
print(f'  Segments included:  {"YES" if INCLUDE_SEGMENTS else "NO"}')


## Cell 5 — Render the HTML pitch

In [ ]:
now = datetime.now().strftime('%d %B %Y')

def kpi_card(label, value, sub=''):
    sub_html = f'<div class="s">{esc(sub)}</div>' if sub else ''
    return f'<div class="card"><div class="l">{esc(label)}</div><div class="v">{esc(value)}</div>{sub_html}</div>'

# Combined hero KPIs
combined_kpis = '<div class="row">' + ''.join([
    kpi_card('Customers', N(combined['customers']), 'unique FNB cardholders'),
    kpi_card('Annual Spend', R(combined['total_spend'])),
    kpi_card('Transactions', N(combined['transactions'])),
    kpi_card('Avg Basket', R(combined['avg_txn_value'])),
    kpi_card('Spend / Customer', R_exact(combined['spend_per_customer']), 'per year'),
    kpi_card('Avg Age', f"{demo['avg_age']}"),
]) + '</div>'

# Per-destination cards
palette = ['#2E75B6', '#16a34a', '#e11d48', '#f59e0b', '#9333ea']

def dest_card(row, colour, idx):
    if row is None:
        return ''
    kpis = [
        kpi_card('Customers', N(row['customers'])),
        kpi_card('Annual Spend', R(row['total_spend'])),
        kpi_card('Avg Basket', R(row['avg_txn_value'])),
        kpi_card('Spend / Customer', R_exact(row['spend_per_customer'])),
    ]
    if 'spend_rank' in row.index and pd.notna(row.get('spend_rank')):
        kpis.append(kpi_card('Category Rank', f"#{int(row['spend_rank'])}", 'in category'))
    if 'penetration_pct' in row.index and pd.notna(row.get('penetration_pct')):
        kpis.append(kpi_card('Customer Reach', f"{row['penetration_pct']:.1f}%", 'of category shoppers'))
    dname = row['DESTINATION']
    bg = f'linear-gradient(180deg,{colour}15 0%,#fff 60%)'
    header = f'<div class="brand-header"><span class="brand-badge" style="background:{colour}">DESTINATION {idx+1}</span><h3>{esc(dname)}</h3></div>'
    return f'<div class="brand-card" style="border-color:{colour};background:{bg}">{header}<div class="row">{"".join(kpis)}</div></div>'

split_cards = ''.join([
    dest_card(row, palette[i % len(palette)], i)
    for i, (_, row) in enumerate(dest_details.iterrows())
])

if COMBINED_OR_SPLIT == 'combined':
    split_section = ''
elif COMBINED_OR_SPLIT == 'split':
    split_section = f'<div class="sec"><h2>How the audience splits</h2>{split_cards}</div>'
else:
    split_section = f'<div class="sec"><h2>How the audience splits between destinations</h2><p class="sub">The combined audience above breaks down into these footprints. Some customers shop at more than one.</p>{split_cards}</div>'

# Category positioning (no competitor names!)
positioning_section = ''
if not dest_details.empty:
    lead = dest_details.iloc[0]
    if 'spend_rank' in lead.index and pd.notna(lead.get('spend_rank')):
        rank_str = f"#{int(lead['spend_rank'])}"
        cat_str = f"in {CATEGORY_TWO}"
        share = f"{lead['market_share_pct']:.1f}%" if pd.notna(lead.get('market_share_pct')) else 'N/A'
        reach = f"{lead['penetration_pct']:.1f}%" if pd.notna(lead.get('penetration_pct')) else 'N/A'
        wallet = f"{lead['avg_share_of_wallet']:.1f}%" if pd.notna(lead.get('avg_share_of_wallet')) else 'N/A'
        rank_card = kpi_card("Category Rank", rank_str, cat_str)
        share_card = kpi_card("Category Share", share, "by spend")
        reach_card = kpi_card("Customer Reach", reach, "of category shoppers")
        wallet_card = kpi_card("Wallet Share", wallet, "of category basket")
        positioning_section = (
            '<div class="sec"><h2>Category positioning</h2>'
            f'<p class="sub">Where {esc(CLIENT_NAME)} sits in {esc(CATEGORY_TWO)} by FNB cardholder spend.</p>'
            '<div class="row">'
            f'{rank_card}{share_card}{reach_card}{wallet_card}'
            '</div></div>'
        )

# Segments section
segments_section = ''
if INCLUDE_SEGMENTS and segments:
    top2_pct = round(sum(s['pct'] for s in segments[:2]), 1)
    segments_section = (
        '<div class="sec"><h2>Customer quality</h2>'
        f'<p class="sub">{esc(CLIENT_NAME)} customers clustered by FNB behavioural segmentation. Segments reflect FNB-wide activity (not client-specific).</p>'
        f'<div class="callout good"><b>{top2_pct}% of {esc(CLIENT_NAME)} customers are in FNB two highest-value segments</b> — a strong indicator of an affluent, engaged audience.</div>'
        '<div class="chbox"><canvas id="chSegments"></canvas></div></div>'
    )

# Trend narrative
trend_narrative = ''
if len(trend) >= 3:
    fh, sh = trend[:len(trend)//2], trend[len(trend)//2:]
    avg_f = sum(m['spend'] for m in fh) / len(fh)
    avg_s = sum(m['spend'] for m in sh) / len(sh)
    d_pct = round(100 * (avg_s - avg_f) / max(avg_f, 1), 1)
    peak = max(trend, key=lambda m: m['spend'])
    trough = min(trend, key=lambda m: m['spend'])
    if d_pct > 3:
        direction = f'<b style="color:#16a34a">growing</b> ({d_pct:+.1f}% H1→H2)'
    elif d_pct < -3:
        direction = f'<b style="color:#e11d48">softening</b> ({d_pct:+.1f}% H1→H2)'
    else:
        direction = f'<b style="color:#334155">stable</b> ({d_pct:+.1f}% H1→H2)'
    trend_narrative = (
        f'{esc(CLIENT_NAME)} monthly spend is {direction}. Peak: '
        f'<b>{esc(peak["month"])}</b> ({R(peak["spend"])}). '
        f'Softest: <b>{esc(trough["month"])}</b> ({R(trough["spend"])}).'
    )

# Activation opportunities
activation_section = ''
if INCLUDE_SEGMENTS and segments:
    ss = {s['segment']: s for s in segment_spend}
    def sz(n): r = next((s for s in segments if s['segment']==n), None); return int(r['customers']) if r else 0
    def sp(n): r = ss.get(n); return int(r['client_annual_spend']) if r else 0
    protect = sz('Champions') + sz('Loyal High Value')
    grow = sz('Steady Mid-Tier')
    reeng = sz('At Risk') + sz('Dormant')
    protect_s = sp('Champions') + sp('Loyal High Value')
    grow_s = sp('Steady Mid-Tier')
    reeng_s = sp('At Risk') + sp('Dormant')
    activation_section = (
        f'<div class="sec"><h2>Activation opportunities</h2>'
        f'<p class="sub">Three pools within the audience, each with a different play and a real spend number attached.</p>'
        f'<div class="act-row">'
        f'<div class="act-card act-protect"><div class="act-badge">PROTECT</div><h3>Champions &amp; Loyal High Value</h3>'
        f'<div class="act-size">{N(protect)}</div><div class="act-money">{R(protect_s)} annual spend at stake</div>'
        f'<div class="act-desc">Your highest-value audience. Retention plays: VIP perks, early product access, premium loyalty tiers.</div></div>'
        f'<div class="act-card act-grow"><div class="act-badge">GROW</div><h3>Steady Mid-Tier</h3>'
        f'<div class="act-size">{N(grow)}</div><div class="act-money">{R(grow_s)} current spend + upside</div>'
        f'<div class="act-desc">Reliable regulars ready to trade up. Basket-builder promotions, category-expansion.</div></div>'
        f'<div class="act-card act-reengage"><div class="act-badge">RE-ENGAGE</div><h3>Dormant &amp; At Risk</h3>'
        f'<div class="act-size">{N(reeng)}</div><div class="act-money">{R(reeng_s)} lapsed spend to recover</div>'
        f'<div class="act-desc">Previously active customers with fading engagement. Win-back offers, personalised comeback campaigns.</div></div>'
        f'</div></div>'
    )



# ── Loyalist section (opt-in) ─────────────────────────────────────
loyalist_section = ''
if loyalist is not None:
    loyalist_section = (
        '<div class="sec" style="background:linear-gradient(180deg,#f0fdf4 0%,#fff 40%); border:2px solid #16a34a">'
        f'<h2 style="color:#166534">Your loyalists</h2>'
        f'<p class="sub">Customers where <b>60%+ of their {esc(CATEGORY_TWO).lower()} basket</b> is spent at {esc(CLIENT_NAME)} — the ones who chose your brand as a way of life. This is your brand armour.</p>'
        '<div class="row">'
        f'{kpi_card("Loyalist customers", N(loyalist["loyalist_customers"]))}'
        f'{kpi_card("Annual spend from loyalists", R(loyalist["loyalist_spend"]))}'
        f'{kpi_card("Regular shoppers", N(loyalist["regular_customers"]), "20-60% of basket")}'
        f'{kpi_card("Occasional shoppers", N(loyalist["occasional_customers"]), "<20% of basket")}'
        '</div>'
        '<div class="callout" style="margin-top:14px">'
        'Loyalists are 5× more valuable than casual shoppers. Every basket-share point you win from Regular → Loyalist is worth thousands of Rands over their lifetime.'
        '</div>'
        '</div>'
    )

# ── Switch opportunity section (opt-in) ───────────────────────────
switch_section = ''
if switch is not None:
    fl_pct = round(100 * int(switch['client_customers']) / max(int(switch['category_customers']), 1), 1)
    non_pct = round(100 * int(switch['non_client_customers']) / max(int(switch['category_customers']), 1), 1)
    switch_section = (
        '<div class="sec" style="background:linear-gradient(180deg,#fff7ed 0%,#fff 40%); border:2px solid #d97706">'
        f'<h2 style="color:#92400e">Switch opportunity — the winnable ground</h2>'
        f'<p class="sub">The size of the {esc(CATEGORY_TWO).lower()} pool in FNB card data — and how much of it is not yet loyal to {esc(CLIENT_NAME)}.</p>'
        '<div class="row">'
        f'{kpi_card("Category customers", N(switch["category_customers"]), f"FNB {CATEGORY_TWO.lower()} shoppers, 12mo")}'
        f'{kpi_card(f"{CLIENT_NAME} customers", N(switch["client_customers"]), f"{fl_pct}% of category")}'
        f'{kpi_card("Non-client customers", N(switch["non_client_customers"]), "the switch pool")}'
        f'{kpi_card("Non-client spend", R(switch["non_client_spend"]), "going to other brands")}'
        '</div>'
        '<div class="callout" style="margin-top:14px">'
        f'{non_pct}% of FNB {esc(CATEGORY_TWO).lower()} shoppers currently spend nothing at {esc(CLIENT_NAME)}. Winning even a small share of that spend represents category growth without cannibalising existing loyalists.'
        '</div>'
        '</div>'
    )

# ── Acquisition target section (opt-in) ───────────────────────────
acq_section = ''
if acq_target is not None:
    avg_prospect = int(acq_target['acq_spend']) / max(int(acq_target['acq_customers']), 1)
    acq_section = (
        '<div class="sec" style="background:linear-gradient(180deg,#eff6ff 0%,#fff 40%); border:2px solid #2E75B6">'
        f'<h2 style="color:#1e40af">Aspirational acquisition target</h2>'
        f'<p class="sub">FNB cardholders who match your ideal buyer — {ACQUISITION_AGE_MIN}-{ACQUISITION_AGE_MAX}, mid-to-upper income, active {esc(CATEGORY_TWO).lower()} spenders — <b>but who don\'t shop at {esc(CLIENT_NAME)} yet</b>. This is the aspirational cohort of the switch pool.</p>'
        '<div class="row">'
        f'{kpi_card("Prospects", N(acq_target["acq_customers"]), f"{ACQUISITION_AGE_MIN}-{ACQUISITION_AGE_MAX}, mid-to-upper income")}'
        f'{kpi_card(f"Annual {CATEGORY_TWO.lower()} spend", R(acq_target["acq_spend"]), "currently going elsewhere")}'
        f'{kpi_card("Avg spend per prospect", R(avg_prospect))}'
        '</div>'
        '<div class="callout" style="margin-top:14px">'
        f'Winning even a small share of this pool means real growth. The audience is already spending — the question is only which brand earns it.'
        '</div>'
        '</div>'
    )


data_obj = {
    'income_gender': [{'band': r['band'], 'male': int(r['male']), 'female': int(r['female'])} for r in income_gender],
    'age_gender':    [{'band': r['band'], 'male': int(r['male']), 'female': int(r['female'])} for r in age_gender],
    'geo':           [{'province': r['province'], 'customers': int(r['customers']), 'spend': float(r['spend'])} for r in geo],
    'trend':         [{'month': r['month'], 'customers': int(r['customers']), 'spend': float(r['spend'])} for r in trend],
    'segments':      [{'name': s['segment'], 'customers': int(s['customers']), 'pct': float(s['pct'])} for s in segments] if INCLUDE_SEGMENTS else [],
}

cross_shop_rows = ''.join(f'<tr><td>{esc(r["category"])}</td><td>{N(r["shoppers"])}</td><td>{R(r["spend"])}</td></tr>' for r in cross_shop)

total_gender = int(demo['male']) + int(demo['female'])
demo_row = ''.join([
    kpi_card('Female', f'{round(100*demo["female"]/total_gender,1)}%', f'{N(demo["female"])} customers'),
    kpi_card('Male',   f'{round(100*demo["male"]/total_gender,1)}%',   f'{N(demo["male"])} customers'),
    kpi_card('Avg age', f'{demo["avg_age"]}', 'years'),
])

CSS = """*{margin:0;padding:0;box-sizing:border-box}
body{font-family:'DM Sans',sans-serif;background:#f8fafc;color:#1a202c}
#hdr{background:linear-gradient(135deg,#0f172a,#1e3a5f);color:#fff;padding:28px 32px}
#hdr h1{font-size:1.9rem;font-weight:700}
#hdr p{opacity:.85;font-size:1rem;margin-top:6px}
#hdr .meta{font-size:.78rem;opacity:.55;margin-top:14px}
.ctn{max-width:1200px;margin:0 auto;padding:24px}
.sec{background:#fff;border-radius:14px;padding:26px 30px;margin:18px 0;border:1px solid #f1f5f9;box-shadow:0 1px 4px rgba(0,0,0,.04)}
.sec h2{font-size:1.35rem;font-weight:700;color:#0f172a;margin-bottom:6px}
.sec .sub{color:#64748b;font-size:.92rem;margin-bottom:18px;line-height:1.5}
.hero{background:linear-gradient(135deg,#fef3c7,#fde68a);border:1px solid #f59e0b}
.hero h2{color:#78350f} .hero .sub{color:#92400e}
.callout{border-radius:10px;padding:14px 18px;margin:12px 0;font-size:.95rem}
.callout.good{background:#dcfce7;border-left:4px solid #16a34a;color:#14532d}
.row{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:12px 0}
.card{background:#f8fafc;border-radius:10px;padding:16px;text-align:center;border-top:3px solid #2E75B6}
.hero .card{background:#fffbeb;border-top-color:#d97706}
.card .l{font-size:.72rem;color:#64748b;font-weight:600;text-transform:uppercase}
.card .v{font-size:1.5rem;font-weight:700;color:#0f172a;margin-top:6px}
.card .s{font-size:.72rem;color:#94a3b8;margin-top:3px}
.chbox{position:relative;height:300px;margin:12px 0}
.chbox.tall{height:360px}
table{width:100%;border-collapse:collapse;margin:12px 0;font-size:.86rem}
th{background:#0f172a;color:#fff;padding:10px 12px;text-align:left;font-size:.72rem;text-transform:uppercase}
td{padding:9px 12px;border-bottom:1px solid #f1f5f9}
.brand-card{background:#fff;border-radius:12px;padding:20px 22px;margin-top:14px;border:2px solid #f1f5f9}
.brand-header{display:flex;align-items:center;gap:10px;margin-bottom:4px}
.brand-header h3{font-size:1.1rem;font-weight:700;color:#0f172a;margin:0}
.brand-badge{font-size:.66rem;font-weight:700;padding:3px 10px;border-radius:12px;color:#fff}
.act-row{display:grid;grid-template-columns:repeat(3,1fr);gap:14px;margin-top:14px}
.act-card{background:#fff;border-radius:12px;padding:20px 22px;border:2px solid #f1f5f9}
.act-card h3{font-size:1.05rem;font-weight:700;color:#0f172a;margin:8px 0 4px}
.act-badge{display:inline-block;font-size:.65rem;font-weight:700;padding:3px 10px;border-radius:12px;color:#fff}
.act-size{font-size:1.7rem;font-weight:700;color:#0f172a;margin:6px 0 0}
.act-money{font-size:.8rem;font-weight:600;color:#78350f;margin:4px 0 10px}
.act-desc{font-size:.85rem;color:#475569;line-height:1.55}
.act-protect{border-color:#16a34a;background:linear-gradient(180deg,#f0fdf4 0%,#fff 60%)} .act-protect .act-badge{background:#16a34a}
.act-grow{border-color:#2E75B6;background:linear-gradient(180deg,#eff6ff 0%,#fff 60%)} .act-grow .act-badge{background:#2E75B6}
.act-reengage{border-color:#e11d48;background:linear-gradient(180deg,#fef2f2 0%,#fff 60%)} .act-reengage .act-badge{background:#e11d48}
.trend-story{background:#f1f5f9;border-radius:10px;padding:14px 18px;margin-top:14px;font-size:.94rem;color:#334155}
"""

JS = """const Data = __DATA__;
const colors = { male:'#2E75B6', female:'#E85C0D', accent:'#1e3a5f', fl:'#f59e0b', seg:['#16a34a','#2E75B6','#f59e0b','#e11d48','#94a3b8'] };
if (typeof ChartDataLabels !== 'undefined') Chart.register(ChartDataLabels);
function mkChart(id, cfg) {
  const el = document.getElementById(id);
  if (!el) return;
  if (!cfg.options) cfg.options = {};
  if (!cfg.options.plugins) cfg.options.plugins = {};
  if (cfg.options.plugins.datalabels === undefined) cfg.options.plugins.datalabels = { display: false };
  new Chart(el, cfg);
}
mkChart('chIncomeGender', { type:'bar', data: { labels: Data.income_gender.map(r=>r.band), datasets: [{ label:'Male', data: Data.income_gender.map(r=>r.male), backgroundColor: colors.male }, { label:'Female', data: Data.income_gender.map(r=>r.female), backgroundColor: colors.female }] }, options: { responsive:true, maintainAspectRatio:false, scales: { y: { beginAtZero:true } } } });
mkChart('chAgeGender', { type:'bar', data: { labels: Data.age_gender.map(r=>r.band), datasets: [{ label:'Male', data: Data.age_gender.map(r=>r.male), backgroundColor: colors.male }, { label:'Female', data: Data.age_gender.map(r=>r.female), backgroundColor: colors.female }] }, options: { responsive:true, maintainAspectRatio:false, scales: { y: { beginAtZero:true } } } });
mkChart('chGeo', { type:'bar', data: { labels: Data.geo.map(r=>r.province), datasets: [{ label:'Spend', data: Data.geo.map(r=>r.spend), backgroundColor: colors.accent }] }, options: { indexAxis:'y', responsive:true, maintainAspectRatio:false, plugins:{legend:{display:false}}, scales: { x: { beginAtZero:true, ticks: { callback: v => 'R' + (v/1e6).toFixed(0) + 'M' } } } } });
mkChart('chTrend', { type:'line', data: { labels: Data.trend.map(r=>r.month), datasets: [{ label:'Spend', data: Data.trend.map(r=>r.spend), borderColor: colors.fl, backgroundColor:'rgba(245,158,11,0.15)', yAxisID:'y', tension:0.3, fill:true }, { label:'Customers', data: Data.trend.map(r=>r.customers), borderColor: colors.accent, backgroundColor:'transparent', yAxisID:'y1', tension:0.3 }] }, options: { responsive:true, maintainAspectRatio:false, scales: { y: { position:'left', ticks:{ callback: v => 'R' + (v/1e6).toFixed(0) + 'M' } }, y1: { position:'right', grid:{ drawOnChartArea:false }, ticks:{ callback: v => (v/1e3).toFixed(0) + 'k' } } } } });
if (Data.segments.length > 0) {
  mkChart('chSegments', { type:'doughnut', data: { labels: Data.segments.map(r => r.name + ' (' + r.pct.toFixed(1) + '%)'), datasets: [{ data: Data.segments.map(r=>r.customers), backgroundColor: Data.segments.map((_,i) => colors.seg[i % colors.seg.length]), borderColor:'#fff', borderWidth:3 }] }, options: { responsive:true, maintainAspectRatio:false, cutout:'58%', plugins: { legend:{position:'right'}, datalabels: { display: (ctx) => { const total = ctx.chart.data.datasets[0].data.reduce((a,b)=>a+b,0); return (ctx.dataset.data[ctx.dataIndex]/total) * 100 >= 8; }, color:'#fff', font:{size:14, weight:'bold'}, formatter: (v, ctx) => { const total = ctx.chart.data.datasets[0].data.reduce((a,b)=>a+b,0); return ((v/total)*100).toFixed(1) + '%'; } } } } });
}
"""

JS_FINAL = JS.replace('__DATA__', json.dumps(data_obj))

html = (
    "<!DOCTYPE html>\n<html lang='en'><head><meta charset='UTF-8'>"
    f"<title>{esc(CLIENT_NAME)} — Audience Snapshot</title>"
    "<script src='https://cdn.jsdelivr.net/npm/chart.js@4.4.0/dist/chart.umd.min.js'></script>"
    "<script src='https://cdn.jsdelivr.net/npm/chartjs-plugin-datalabels@2.2.0'></script>"
    "<style>@import url('https://fonts.googleapis.com/css2?family=DM+Sans:wght@300;400;500;600;700&display=swap');"
    + CSS +
    "</style></head><body>"
    f"<div id='hdr'><h1>{esc(CLIENT_NAME)} — Audience Snapshot</h1>"
    f"<p>{esc(CLIENT_CONTEXT)}</p>"
    f"<div class='meta'>Generated {esc(now)} · Rolling 12 months</div></div>"
    "<div class='ctn'>"
    f"<div class='sec hero'><h2>The reach story</h2><p class='sub'>Distinct FNB cardholders shopping at {esc(CLIENT_NAME)} in the last 12 months.</p>{combined_kpis}</div>"
    + split_section
    + positioning_section
    + segments_section
    + f"<div class='sec'><h2>Who they are</h2><div class='row'>{demo_row}</div></div>"
    + "<div class='sec'><h2>Income &amp; gender profile</h2><div class='chbox tall'><canvas id='chIncomeGender'></canvas></div></div>"
    + "<div class='sec'><h2>Age &amp; gender profile</h2><div class='chbox tall'><canvas id='chAgeGender'></canvas></div></div>"
    + f"<div class='sec'><h2>Geographic footprint</h2><p class='sub'>Total {esc(CLIENT_NAME)} spend by province.</p><div class='chbox tall'><canvas id='chGeo'></canvas></div></div>"
    + f"<div class='sec'><h2>Monthly trend</h2><p class='sub'>{esc(CLIENT_NAME)} spend and customer counts month-over-month.</p><div class='chbox tall'><canvas id='chTrend'></canvas></div><div class='trend-story'>{trend_narrative}</div></div>"
    + activation_section
    + loyalist_section
    + switch_section
    + acq_section
    + f"<div class='sec'><h2>Adjacent spend — bundling &amp; co-brand opportunities</h2><p class='sub'>The top non-{esc(CATEGORY_TWO).lower()} categories your customers spend in.</p><table><tr><th>Category</th><th>Shoppers</th><th>Annual spend</th></tr>{cross_shop_rows}</table></div>"
    + "</div><script>" + JS_FINAL + "</script></body></html>"
)

with open(OUTPUT_HTML, 'w') as f:
    f.write(html)

print()
print(f'✓ Pitch written to: {OUTPUT_HTML}')
print()
print('═══ Scope-safety checklist ═══')
print(f'  Client:               {CLIENT_NAME}')
print(f'  DESTINATIONS:         {DESTINATIONS}')
print(f'  Combined = union:     ✓ validated within 1%')
print(f'  stg_transactions sum: never for totals (uses int_customer_category_spend)')
print(f'  Rounding for /cust:   R_exact (no R4k rounding)')
print(f'  Segments shown:       {"YES with caveat" if INCLUDE_SEGMENTS else "NO"}')
print(f'  Competitor names:     never (Category Positioning shows only your rank + share)')
print('══════════════════════════════')


## Cell 6 — Preview inline (optional)

In [ ]:
from IPython.display import IFrame, display
display(IFrame(OUTPUT_HTML, width='100%', height=800))
